# 第 19 章：完整领域模型工程模板

这个 notebook 对应 `lessons/19_domain_model_template.md`，演示如何校验领域模型模板目录、解析配置、生成 run manifest、检查报告链路，并用 release gate 阻止缺报告或缺 rollback target 的版本。

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from src.domain_model_template.template import (
    DomainConfig,
    RunManifest,
    check_release_readiness,
    parse_simple_yaml,
    validate_config,
    validate_report_chain,
    validate_template_tree,
    write_run_manifest,
)

## 1. 模板目录

模板必须包含 configs、data、scripts、src、tests、reports。

In [ ]:
template_root = Path("projects/domain_model_template")
validate_template_tree(template_root)
sorted(path.name for path in template_root.iterdir() if path.is_dir())

## 2. 配置解析

教学版 parser 支持本模板使用的简单 YAML 子集，避免额外依赖。

In [ ]:
config_text = (template_root / "configs" / "data.yaml").read_text()
config = DomainConfig.from_dict(parse_simple_yaml(config_text))
validate_config(config)
config.to_dict()

## 3. Run Manifest

`run_manifest.json` 把模型、数据、RAG index、配置和报告产物连成证据链。

In [ ]:
manifest = RunManifest(
    run_id="template_run_v1",
    base_model="tiny-base@main",
    dataset_version="sft_v1",
    eval_dataset_version="eval_v1",
    rag_index_version="kb_v1",
    model_version="domain-model-v1",
    config_files=["configs/data.yaml", "configs/eval.yaml", "configs/serving.yaml"],
    report_files=[
        "reports/data_quality_report.md",
        "reports/eval_report.md",
        "reports/failure_cases.csv",
        "reports/risk_report.md",
        "reports/model_card.md",
    ],
)

with TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / "run_manifest.json"
    write_run_manifest(path, manifest)
    print(path.read_text())

## 4. 报告链路与发布门禁

eval report、model card、risk report 需要互相引用；发布检查会阻止缺关键产物的版本。

In [ ]:
validate_report_chain(template_root, manifest)
release = check_release_readiness(template_root, config, manifest)
release